# 成交量激增与耀眼5分钟事件研究复现

本 Notebook 是项目的干净复现展示版本。

注意：由于数据授权限制，本项目不公开原始 A 股 1 分钟行情数据。原始分钟数据来自米筐 RQData。

本 Notebook 主要读取已经导出的统计表和图表，用于展示复现结果。


## 1. 项目结构

GitHub 项目主要包含：

- figures/：核心复现图表
- tables/：核心统计表和事件样本数据
- README.md：项目说明
- notebooks/reproduction_main_clean.ipynb：干净版复现展示 Notebook

未上传内容：

- 原始 1 分钟行情数据
- 全市场完整分钟 parquet 文件
- 中间 batch 文件
- 米筐运行过程中的错误输出和调试代码


## 2. 事件定义

对每只股票、每个交易日，定义成交量激增倍数：

spike_ratio = 当前1分钟成交量 / 过去20个交易日同一分钟成交量中位数

每只股票每天选择 spike_ratio 最大的分钟作为成交量激增事件时刻。

围绕事件时刻构造三个窗口：

- pre5：事件发生前5分钟
- event5：事件发生当分钟开始的连续5分钟，即耀眼5分钟
- post5：事件结束后的连续5分钟


In [ ]:
# 读取核心统计表
import os
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

TABLE_DIR = os.path.join(BASE_DIR, 'tables')
FIG_DIR = os.path.join(BASE_DIR, 'figures')

summary_path = os.path.join(TABLE_DIR, 'summary_statistics_2024Q1Q3.csv')
time_dist_path = os.path.join(TABLE_DIR, 'event_time_distribution_2024Q1Q3.csv')
monthly_summary_path = os.path.join(TABLE_DIR, 'monthly_summary_2024Q1Q3.csv')
sample_path = os.path.join(TABLE_DIR, 'volume_spike_events_2024Q1Q3_sample10000.csv')

print('TABLE_DIR:', TABLE_DIR)
print('FIG_DIR:', FIG_DIR)
print('summary exists:', os.path.exists(summary_path))
print('time_dist exists:', os.path.exists(time_dist_path))
print('monthly_summary exists:', os.path.exists(monthly_summary_path))
print('sample exists:', os.path.exists(sample_path))


In [ ]:
# 查看总体统计结果
summary_df = pd.read_csv(summary_path)
summary_df


In [ ]:
# 转置展示 summary，便于阅读
summary_df.T


In [ ]:
# 查看事件时刻分布表
time_dist = pd.read_csv(time_dist_path)
print(time_dist.shape)
time_dist.head()


In [ ]:
# 查看月度统计表
monthly_summary = pd.read_csv(monthly_summary_path)
monthly_summary


In [ ]:
# 查看事件样本数据
event_sample = pd.read_csv(sample_path)
print(event_sample.shape)
event_sample.head()


## 3. 核心图表

下面展示四张复现图：

1. 日内成交量激增事件分布
2. 耀眼5分钟平均收益
3. 月度事件数量
4. 事件前后收益对比


In [ ]:
from IPython.display import Image, display

fig_files = [
    'intraday_event_count.png',
    'intraday_event5_return.png',
    'monthly_event_count.png',
    'mean_return_pre_event_post.png'
]

for fname in fig_files:
    fpath = os.path.join(FIG_DIR, fname)
    print(fname, os.path.exists(fpath))
    if os.path.exists(fpath):
        display(Image(filename=fpath))


## 4. 结果解释

从复现结果可以观察到：

1. 成交量激增事件在日内呈现明显的时间分布特征。
2. 耀眼5分钟窗口中，收益率和波动率表现与事件前后窗口存在差异。
3. 事件后窗口可以用于观察短期价格延续或反转效应。
4. 月度事件数量反映了样本覆盖度和市场交易活跃度的变化。

这些结果可以作为后续深入事件研究、分组检验和稳健性分析的基础。


## 5. 数据授权说明

本项目不上传原始 A 股 1 分钟行情数据。

原因：

- 原始数据来自米筐 RQData，存在数据授权限制。
- 原始分钟数据体积较大，不适合上传 GitHub。

如果需要完全复现，需要用户自行具备 RQData 权限，并根据项目代码重新下载数据。
